In [ ]:
INDEX = 8  # Select which questions in the benchmark to test
DATA_SOURCE = "archeology"

# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
from os import environ
from sys import path

from torch.backends import cudnn

# enforce more deterministic behavior
environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

path.append("../../../src")

from processor.core.ir_system.lm_interface import LMInterface
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.model.interface.impl.gpt import GPT

from tqdm import tqdm
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption


from logging import INFO
import os
import json

In [ ]:
gpt = GPT("gpt-4o-mini")

In [ ]:
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl("../../../benchmark/benchmark_archeology.jsonl")
INITIAL_PROMPT = benchmark[INDEX]["interactive_initial_prompt"]
benchmark[INDEX]

In [ ]:
from processor.core.ir_system.ir_data_model import AbstractDocument, convert_retrieval_results_to_str


def get_format_to_gpt(tables: list[AbstractDocument]):
    return f"SYSTEM OUTPUT:\n```{convert_retrieval_results_to_str(tables)}```"

def get_initial_prompt_to_chatgpt(domain: str, question: str):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system supports static table lookup: it returns one or more relevant tables based on your description. However:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- The system does not explain its reasoning—it simply returns tables for you to explore.

Your task is to gradually explore and refine your question about some aspect of the data. You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive. You will gradually refine your question by examining the contents of the tables returned.

In this scenario:
- The system already has access to an internal dataset.
- You are familiar with the domain and have seen similar datasets before.
- You are not uploading new datasets or asking if data exists — you assume it does.

Here is a possible eventual goal (you do not know this at the start, and you may or may not reach it):

{question}

Your behavior should reflect:
- You are familiar with the domain but must infer relevant relationships from static tables.
- You refine your question step-by-step depending on what the returned tables show.
- You may explore tangents or ask for different tables in later turns.
- You will only reach the specific question above if you deduce it from the tables, which may take multiple turns.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}"""

In [ ]:
logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
llm_path = "../../../src/processor/model/weight/qwen3-8b"
embed_model_path = "../../../src/processor/model/weight/bge-base"
pneuma = LMInterface({
    "llm": get_llm(llm_path)(llm_path),
    "embed_model": get_embed_model()(embed_model_path),
}, logger)

# Evaluation

In [ ]:
benchmark[INDEX]["original_direct_question"]

In [ ]:
ITERATION_LIMIT = 15

gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATA_SOURCE, benchmark[INDEX]["original_direct_question"]
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
for iteration in tqdm(range(ITERATION_LIMIT)):
    tables = pneuma.retrieve_documents(
        INITIAL_PROMPT,
        [DATA_SOURCE],
        10,
        [RetrieverType.PNEUMA],
    )[RetrieverType.PNEUMA]
    format_to_gpt = get_format_to_gpt(tables)
    if iteration == 0:
        gpt_messages[0]['content'] += f"\n{format_to_gpt}"
    else:
        gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
    updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
    gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
    if updated_user_prompt.startswith("YOU:"):
        updated_user_prompt = updated_user_prompt[4:]
        updated_user_prompt = updated_user_prompt.strip()
    curr_user_prompt = updated_user_prompt
    print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

In [ ]:
write_jsonl(f"benchmark_data/{DATA_SOURCE}_{INDEX+1}.jsonl", gpt_messages, True)